In [25]:
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"


v = embed.encode(q)
print(len(v))
print(v[0])

384
-0.020582036807885073


## Loading the data 

In [26]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [32]:


docs_by_id = {doc["filename"]: doc for doc in documents}

doc_id = "02-vector-search/lessons/07-sqlitesearch-vector.md"
v_doc = embed.encode(docs_by_id[doc_id]["content"])

print(v.dot(v_doc))


0.361070280302606


In [30]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


In [ ]:
chunk_texts = [chunk["content"] for chunk in chunks]

X = embed.encode_batch(chunk_texts)
scores = X.dot(v) 



In [39]:
import numpy as np

best_idx = np.argmax(scores)
best_chunk = chunks[best_idx]
print(best_chunk["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [40]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [41]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [43]:
print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


In [ ]:
pgq= "How do I store vectors in PostgreSQL?"
v_pgq = embed.encode(pgq)

vector_results = vindex.search(v_pgq, num_results=5)

for i in range(0, 5):
    print(vector_results[i]["filename"])


02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [48]:
from minsearch import Index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(chunks)
text_results = index.search(pgq, num_results=5)
for i in range(0, 5):
    print(text_results[i]["filename"])


02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [49]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [52]:
print(results[0]["filename"])

02-vector-search/lessons/08-pgvector.md


In [53]:
q2= "How do I give the model access to tools?"
vq2 = embed.encode(q2)
vector_results2 = vindex.search(vq2, num_results=5)
text_results2 = index.search(q2, num_results=5)
results = rrf([vector_results2, text_results2])
print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
